In [24]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import time
import re
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def get_article_details(url):
    """Extrae el cuerpo y la fecha real desde adentro del post."""
    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
        res = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(res.text, 'html.parser')
        
        # 1. FECHA: Buscamos el patrón de texto de fecha (ej: "March 25, 2026")
        # Bitovi suele poner la fecha en un span o div cerca del título
        date_text = ""
        date_element = soup.find('time') or soup.select_one('span.date, .post-date, .blog-post-date')
        
        if date_element:
            date_text = date_element.get_text().strip()
        else:
            # Fallback: buscar un patrón de fecha en todo el texto (Mes Día, Año)
            match = re.search(r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}', soup.text)
            if match:
                date_text = match.group(0)

        # Normalización de fecha
        final_date = "2026-01-01"
        if date_text:
            try:
                # Limpiar texto extra como "Published on"
                clean_date = date_text.replace("Published on", "").strip()
                dt = datetime.strptime(clean_date, "%B %d, %Y")
                final_date = dt.strftime("%Y-%m-%d")
            except: pass

        # 2. CONTENIDO: Selector basado en la estructura de HubSpot/Bitovi
        content_div = soup.select_one('.blog-post-body') or soup.select_one('.post-body') or soup.find('article')
        content = ""
        if content_div:
            for tag in content_div(["script", "style", "nav", "footer"]): tag.decompose()
            content = content_div.get_text(separator=' ', strip=True)
            
        return content, final_date
    except:
        return "", "2026-01-01"

def scrape_bitovi_blog(max_pages=50): # Agregamos parámetro de páginas
    base_url = "https://www.bitovi.com"
    seen_links = set()
    docs = []

    for page_num in range(1, max_pages + 1):
        # Construimos la URL de la página actual
        blog_url = f"{base_url}/blog/page/{page_num}"
        print(f"\n--- Escaneando Página {page_num}: {blog_url} ---")
        
        headers = {"User-Agent": "Mozilla/5.0"}
        try:
            response = requests.get(blog_url, headers=headers, timeout=15)
            if response.status_code != 200:
                print(f"Llegamos al final o hubo un error en la página {page_num}")
                break
            
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Buscamos todos los links a posts
            all_links = soup.find_all('a', href=True)
            page_posts = []

            for a in all_links:
                href = a['href']
                # Filtro para identificar artículos reales
                if '/blog/' in href and not any(x in href for x in ['/tag/', '/author/', '/topic/', 'page/']):
                    full_url = f"{base_url}{href}" if href.startswith('/') else href
                    if full_url not in seen_links:
                        title = a.get_text().strip()
                        if len(title) > 15: # Evitar links cortos como "Read More"
                            page_posts.append((full_url, title))
                            seen_links.add(full_url)

            if not page_posts:
                print("No se encontraron más artículos en esta página.")
                break

            print(f"Se encontraron {len(page_posts)} nuevos artículos en la página {page_num}.")

            # Procesamos cada link de esta página
            for url, title in page_posts:
                print(f"  -> Extrayendo contenido profundo: {title[:40]}...")
                content, date_iso = get_article_details(url)
                if content:
                    docs.append(Document(
                        page_content=content,
                        metadata={"title": title, "source": url, "date": date_iso}
                    ))
                time.sleep(0.5) # Delay para no ser bloqueados

        except Exception as e:
            print(f"Error en la página {page_num}: {e}")
            break
            
    return docs

def ingest_data():
    raw_documents = scrape_bitovi_blog()
    if not raw_documents:
        print("No se encontraron documentos.")
        return

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150)
    documents = text_splitter.split_documents(raw_documents)
    
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    
    # IMPORTANTE: Borrar la DB anterior si existe para no mezclar fechas viejas
    persist_directory = "./chroma_db"
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory=persist_directory
    )
    print(f"¡Ingesta completada! {len(raw_documents)} artículos indexados con fechas reales.")

if __name__ == "__main__":
    ingest_data()


--- Escaneando Página 1: https://www.bitovi.com/blog/page/1 ---
Se encontraron 12 nuevos artículos en la página 1.
  -> Extrayendo contenido profundo: Figma's MCP Just Gave AI Agents 'Write' ...
  -> Extrayendo contenido profundo: Extend Role-Based Authentication in Self...
  -> Extrayendo contenido profundo: How to Build Flexible Table Components i...
  -> Extrayendo contenido profundo: Slots have transformed Figma’s ability t...
  -> Extrayendo contenido profundo: How to Build KPI Trees with AI: A Miro S...
  -> Extrayendo contenido profundo: How to Achieve a 50% Productivity Boost ...
  -> Extrayendo contenido profundo: Production Ready AI Agents: Making LangC...
  -> Extrayendo contenido profundo: Bitovi Is Now a Certified Figma Service ...
  -> Extrayendo contenido profundo: Quick Start Guide to Self-Hosting Nx Clo...
  -> Extrayendo contenido profundo: Bitovi & Miro: Collaboration, Supercharg...
  -> Extrayendo contenido profundo: From JavaScript to Digital Delivery: Bit...
  ->

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


¡Ingesta completada! 484 artículos indexados con fechas reales.


In [8]:
# Suponiendo que ya tenés el objeto 'vectorstore' creado
# Si no, cargalo rápido:
# vectorstore = Chroma(persist_directory=PERSIST_DIR, embedding_function=embeddings)

# Obtenemos los IDs de los documentos indexados
persist_directory = "../data/chroma_db"
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Conectar a la base de datos
vectorstore = Chroma(
    persist_directory=persist_directory, 
    embedding_function=embeddings
)

data = vectorstore.get(include=['metadatas'])
metadatas = data['metadatas']

if metadatas:
    print(f"Total de fragmentos en la base: {len(metadatas)}")
    print("\nEjemplo de metadata del primer fragmento:")
    import json
    print(json.dumps(metadatas[0], indent=2))
    
    # Verificamos qué campos únicos existen en toda la base
    keys = set()
    for m in metadatas:
        keys.update(m.keys())
    print(f"\nCampos disponibles en la metadata: {keys}")
else:
    print("La base de datos parece estar vacía.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total de fragmentos en la base: 4583

Ejemplo de metadata del primer fragmento:
{
  "source": "https://bitovi.com/blog/figma-just-opened-the-canvas-to-agents.-heres-what-actually-happens",
  "date": "2026-03-25",
  "title": "Figma's MCP Just Gave AI Agents 'Write' Access. A First Look at What It Can Do.          March 25, 2026    Figma recently announced that AI agents can now write directly to Figma's canvas through their MCP server. It's a big deal. For designers and developers who've struggled to keep code and designs in sync, the idea of an agent that can generate components the right way, apply variables, and work within your design system is the kind of thing that sounds too good to be true.      Ali Shouman"
}

Campos disponibles en la metadata: {'date', 'title', 'source'}


In [2]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Configuración de acceso
persist_directory = "../data/chroma_db"
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Conectar a la base de datos
vectorstore = Chroma(
    persist_directory=persist_directory, 
    embedding_function=embeddings
)

# 3. Obtener todos los registros
# Usamos .get() para traer la información sin realizar búsqueda semántica
data = vectorstore.get()

if not data['ids']:
    print("La base de datos Chroma está vacía.")
else:
    # 4. Estructurar para ordenar
    # Combinamos metadatos y IDs para procesar
    records = []
    for i in range(len(data['ids'])):
        records.append({
            "metadata": data['metadatas'][i],
            "id": data['ids'][i]
        })

    # 5. Ordenar por fecha (descendente: de más reciente a más antiguo)
    # Si 'date' no existe, usamos una fecha mínima para enviarlo al final
    records_sorted = sorted(
        records, 
        key=lambda x: x['metadata'].get('date', '0000-00-00'), 
        reverse=True
    )

    print(f"--- Documentos en Chroma: {len(records)} fragmentos encontrados ---\n")
    
    # Para evitar repetir el título de artículos que tienen varios chunks
    articulos_vistos = set()

    for item in records_sorted:
        meta = item['metadata']
        titulo = meta.get('title', 'Sin título')
        fecha = meta.get('date', 'Fecha desconocida')
        fuente = meta.get('source', 'Sin URL')

        # Agrupamos por título para que la consola sea legible
        if titulo not in articulos_vistos:
            print(f"📅 [{fecha}] - {titulo}")
            print(f"   🔗 {fuente}")
            print("-" * 60)
            articulos_vistos.add(titulo)

    print(f"\nTotal de artículos únicos: {len(articulos_vistos)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- Documentos en Chroma: 4583 fragmentos encontrados ---

📅 [2026-03-25] - Figma's MCP Just Gave AI Agents 'Write' Access. A First Look at What It Can Do.          March 25, 2026    Figma recently announced that AI agents can now write directly to Figma's canvas through their MCP server. It's a big deal. For designers and developers who've struggled to keep code and designs in sync, the idea of an agent that can generate components the right way, apply variables, and work within your design system is the kind of thing that sounds too good to be true.      Ali Shouman
   🔗 https://bitovi.com/blog/figma-just-opened-the-canvas-to-agents.-heres-what-actually-happens
------------------------------------------------------------
📅 [2026-03-10] - Extend Role-Based Authentication in Self-Hosted Temporal with Helm          March 10, 2026    Repo: temporal-auth (helm-support branch)      Matt Chaffe
   🔗 https://bitovi.com/blog/extend-role-based-authentication-in-self-hosted-temporal-with-helm
-

In [12]:
import os
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM

# Imports específicos solicitados
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever

# Para que el modelo local no alucine con la sintaxis de Chroma
from langchain_community.query_constructors.chroma import ChromaTranslator

# Configuración de rutas (subiendo un nivel desde src/ a data/)
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()
PERSIST_DIR = os.path.normpath(os.path.join(BASE_DIR, "..", "data", "chroma_db"))


class CleanSelfQueryRetriever(SelfQueryRetriever):
    """Retriever que limpia los filtros mal formateados por modelos locales."""
    
    def _get_docs_with_query(self, query: str, search_kwargs: dict) -> list:
        # Extraemos el filtro 'where' de Chroma
        if "filter" in search_kwargs:
            f = search_kwargs["filter"]
            # Si Llama mandó {'date': {'date': '...', 'type': 'date'}}
            # lo simplificamos a {'date': '...'}
            for key, value in f.items():
                if isinstance(value, dict) and key in value:
                    f[key] = value[key]
        
        return super()._get_docs_with_query(query, search_kwargs)

def get_bitovi_retriever():
    # 1. Configurar Embeddings (debe ser el mismo que usaste en el indexer)
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    
    # 2. Conectar con la base de datos persistente
    vectorstore = Chroma(
        persist_directory=PERSIST_DIR, 
        embedding_function=embeddings
    )

    # 3. Definir la metadata para que el LLM sepa qué filtrar
    metadata_field_info = [
    AttributeInfo(
        name="date",
        description="The date in YYYY-MM-DD string format. NEVER use objects or dictionaries, ONLY plain strings.",
        type="string", 
    ),
    AttributeInfo(
        name="title",
        description="The plain text title of the article.",
        type="string",
    ),
    AttributeInfo(
        name="source",
        description="The URL link of the article.",
        type="string",
    ),
]

    document_content_description = "Technical blog posts from Bitovi about AI, DevOps, Frontend, and UX."
    
    # 4. Configurar Llama 3.1 vía Ollama como el 'cerebro' del retriever
    llm = OllamaLLM(model="llama3.1:8b", temperature=0)

    # 5. Crear el retriever con el traductor de Chroma
    # Usamos nuestra clase personalizada en lugar de la original
    retriever = CleanSelfQueryRetriever.from_llm(
        llm,
        vectorstore,
        document_content_description,
        metadata_field_info,
        structured_query_translator=ChromaTranslator(),
        verbose=True
    )
    return retriever



In [13]:
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

llm = OllamaLLM(model="llama3.1:8b", temperature=0)

# 1. Definimos un prompt que obligue al modelo a citar sus fuentes
template = """
You are a helpful assistant for Bitovi. Use the following pieces of context to answer the question.
If the answer is in the context, provide the answer and ALWAYS list the Title, Date, and Source URL at the end.

Context: {context}
Question: {question}

Helpful Answer (include metadata):"""

custom_prompt = PromptTemplate(template=template, input_variables=["context", "question"])

# 2. Creamos la cadena
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=get_bitovi_retriever(),
    chain_type_kwargs={"prompt": custom_prompt}
)

# 3. Ejecutamos
response = qa_chain.invoke("What is the latest post about DevOps?")
print(response["result"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ValueError: Expected where operand value to be a str, int, float, bool, or list of those type, got {'date': '2022-01-01', 'type': 'date'} in query.

In [24]:
import os
from langchain.tools import tool
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Suponiendo que ya tenés PERSIST_DIR y embeddings configurados
@tool
def retrieve_bitovi_docs(query: str, k: int = 5):
    """
    Recupera artículos de Bitovi. 
    Si la consulta pide lo 'último' o 'reciente', ordena cronológicamente.
    Si pide un tema, usa búsqueda semántica.
    """
    print(f"\n[TOOL] Procesando query: {query}")
    
    # 1. Traemos TODOS los metadatos (o una cantidad grande) para ordenar en memoria
    # Esto es el equivalente a un SELECT * FROM metadata
    data = vectorstore.get(include=['metadatas', 'documents'])
    
    all_docs = []
    for i in range(len(data['documents'])):
        # Reconstruimos el formato de documento de LangChain
        all_docs.append({
            "content": data['documents'][i],
            "metadata": data['metadatas'][i]
        })

    # 2. Lógica de Decisión
    is_chrono_request = any(w in query.lower() for w in ["latest", "recent", "nuevo", "ultimo", "actualidad"])

    if is_chrono_request:
        # ORDEN PURAMENTE CRONOLÓGICO (Determinista)
        print("[LOG] Aplicando ordenamiento por fecha DESC")
        sorted_docs = sorted(
            all_docs, 
            key=lambda x: str(x['metadata'].get('date', '0000-00-00')), 
            reverse=True
        )
        results = sorted_docs[:k]
    else:
        # BÚSQUEDA SEMÁNTICA (Si pregunta por un tema como "DevOps")
        print("[LOG] Aplicando búsqueda por similitud")
        # Aquí usamos el similarity_search normal
        results_raw = vectorstore.similarity_search(query, k=k)
        results = [{"content": d.page_content, "metadata": d.metadata} for d in results_raw]

    # 3. Formateo de salida para el LLM
    output = []
    for i, doc in enumerate(results, 1):
        m = doc['metadata']
        output.append(f"Document {i}:\nTitle: {m.get('title')}\nDate: {m.get('date')}\nSource: {m.get('source')}\nContent: {doc['content'][:300]}...")

    return "\n\n".join(output)

In [16]:
from langchain_ollama import ChatOllama

# Definimos el modelo con capacidad de usar herramientas
llm = ChatOllama(model="llama3.1:8b", temperature=0)
llm_with_tools = llm.bind_tools([retrieve_bitovi_docs])

# Ahora el agente decidirá cuándo llamar a la función
response = llm_with_tools.invoke("What are the 3 latest posts about AI?")

response

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-04-20T17:08:49.882294402Z', 'done': True, 'done_reason': 'stop', 'total_duration': 39124529040, 'load_duration': 38263568292, 'prompt_eval_count': 204, 'prompt_eval_duration': 111975744, 'eval_count': 40, 'eval_duration': 718721422, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019dabdd-277f-7680-95cb-c97a551c5279-0', tool_calls=[{'name': 'retrieve_bitovi_docs', 'args': {'query': "AI AND (title OR content) AND published_date > '2022-01-01' LIMIT 3"}, 'id': '39d41c19-e0cd-416d-a229-fa34b608b48d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 204, 'output_tokens': 40, 'total_tokens': 244})

In [ ]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Definimos el modelo
llm = OllamaLLM(model="llama3.1:8b", temperature=0)

# 2. El Prompt: Aquí es donde le damos la "personalidad" y la instrucción de los metadatos
prompt = ChatPromptTemplate.from_template("""
Eres un experto técnico de Bitovi. Tu tarea es responder preguntas usando el contexto proporcionado.

REGLAS CRÍTICAS:
1. El contexto ya viene ordenado por fecha de lo más reciente a lo más antiguo.
2. DEBES mantener este orden estrictamente en tu respuesta.
3. No ignores las fechas; el artículo con la fecha más futura es el más reciente.

CONTEXTO:
{context}

PREGUNTA:
{question}

RESPUESTA:
""")

def test_answer(user_query):
    print(f"\n--- Probando Query: '{user_query}' ---")
    
    # 3. Llamamos a tu herramienta manualmente para ver qué trae
    # (Asegurate de tener importada la tool 'retrieve_bitovi_docs')
    context_docs = retrieve_bitovi_docs.invoke({"query": user_query, "k": 3})
    
    # 4. Construimos la cadena
    chain = prompt | llm | StrOutputParser()
    
    # 5. Invocamos
    response = chain.invoke({
        "context": context_docs,
        "question": user_query
    })
    
    print("\n--- RESPUESTA DEL MODELO ---")
    print(response)

if __name__ == "__main__":
    # Prueba 1: Búsqueda de actualidad (Ordenamiento por fecha)
    test_answer("What are the most recent articles?")
    
    # Prueba 2: Búsqueda específica de tecnología
    # test_answer("Tell me about DevOps best practices mentioned in the blog.")


--- Probando Query: 'What are the most recent articles?' ---

[TOOL] Procesando query: What are the most recent articles?
[LOG] Aplicando ordenamiento por fecha DESC

--- RESPUESTA DEL MODELO ---
Según el contexto proporcionado, los artículos más recientes son:

1. Document 3: "Figma's MCP Just Gave AI Agents 'Write' Access. A First Look at What It Can Do." (Fecha: 2026-03-25)
2. Document 2: "Figma's MCP Just Gave AI Agents 'Write' Access. A First Look at What It Can Do." (Fecha: 2026-03-25)
3. Document 1: "Figma's MCP Just Gave AI Agents 'Write' Access. A First Look at What It Can Do." (Fecha: 2026-03-25)

Estos tres artículos tienen la misma fecha y título, pero Document 3 proporciona más detalles sobre las capacidades de los agentes de Figma en cuanto a la creación de componentes y la reestructuración del canvas.
